# 🧮 Naive Bayes
**ISLP Chapter 4 · Pattern Recognition for the Rest of Us**

> Naive Bayes extends the Bayesian classification idea with one strong simplification: assume features are conditionally independent given the class. Despite this "naive" assumption often being wrong, it works surprisingly well in practice — especially for text.

### What you'll learn
- The conditional independence assumption and why it's "naive"
- Gaussian Naive Bayes for numeric features
- Multinomial Naive Bayes for text classification
- Why NB works well even when the independence assumption is violated
- NB vs logistic regression — when to use each

### Dataset: SMS Spam (text classification)

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline

## 📐 Part 1 — Bayes with Independence Assumption

Full Bayes would require estimating P(X₁=x₁, X₂=x₂, ..., Xₚ=xₚ | Y=k) — the joint distribution of all features. With p=1000 features this is impossible.

**Naive Bayes** assumes conditional independence:
```
P(X₁,...,Xₚ | Y=k) = P(X₁|Y=k) × P(X₂|Y=k) × ... × P(Xₚ|Y=k)
```

Now we only need to estimate p separate univariate distributions — completely tractable.

The "naive" part: features are rarely truly independent. But the resulting classifier is surprisingly robust — the probability estimates may be wrong, but the *decision* (which class gets the highest probability) is often right.

In [ ]:
# Wine Quality dataset — predict whether a wine is high quality (score >= 7)
# Real sensory + chemical measurements from Portuguese wines
# Much more meaningful than Iris for business analytics students

import pandas as pd
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from scipy.stats import norm
import matplotlib.pyplot as plt

try:
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
    wine = pd.read_csv(url, sep=';')
    print("Loaded Wine Quality dataset from UCI")
except:
    from sklearn.datasets import load_wine
    data = load_wine(as_frame=True)
    wine = data.frame.rename(columns={'target': 'quality'})
    wine['quality'] = wine['quality'] * 3  # scale to 0-12 range
    print("Using sklearn wine dataset (fallback)")

# Binary target: quality >= 7 = "high quality"
wine['high_quality'] = (wine['quality'] >= 7).astype(int)
feature_cols = [c for c in wine.columns if c not in ['quality','high_quality']]
X_wine = wine[feature_cols]
y_wine = wine['high_quality']

print(f"Wine dataset: {wine.shape}")
print(f"High quality (score >= 7): {y_wine.mean():.1%}")
print(f"Features: {feature_cols}")
wine.head(3)


In [ ]:
# Gaussian NB on Wine Quality — show what it learns
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from scipy.stats import norm

X_wine_arr = X_wine.values
X_tr, X_te, y_tr, y_te = train_test_split(X_wine_arr, y_wine,
                                            test_size=0.25, random_state=42,
                                            stratify=y_wine)
gnb = GaussianNB()
gnb.fit(X_tr, y_tr)

print("=== Gaussian NB: Wine Quality ===")
print(classification_report(y_te, gnb.predict(X_te),
                            target_names=['Standard','High Quality']))

# Visualize: what distributions GNB learned for top features
top_features = ['alcohol', 'volatile acidity', 'sulphates']
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, feat in zip(axes, top_features):
    feat_idx = list(X_wine.columns).index(feat)
    for cls, color, label in [(0,'#1e5fa8','Standard (<=6)'),
                               (1,'#e85d20','High Quality (>=7)')]:
        mu  = gnb.theta_[cls, feat_idx]
        std = np.sqrt(gnb.var_[cls, feat_idx])
        x_range = np.linspace(mu - 3.5*std, mu + 3.5*std, 200)
        ax.plot(x_range, norm.pdf(x_range, mu, std), color=color, lw=2, label=label)
        data_cls = X_wine.values[y_wine==cls, feat_idx]
        ax.hist(data_cls, bins=20, color=color, alpha=0.2, density=True)
    ax.set_title(feat)
    ax.legend(fontsize=8)

plt.suptitle("Gaussian NB: Learned distributions per class\n(separation = predictive power)",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print("\n📌 Alcohol content: high-quality wines cluster at higher values")
print("   Volatile acidity: high-quality wines have LOWER acidity (less vinegary)")
print("   Where the distributions overlap least = most discriminating feature")


In [ ]:
# Text classification: spam detection
X_text = sms['message'].values
y_text = sms['spam'].values

# Pipeline: TF-IDF → Multinomial NB
pipeline_mnb = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english')),
    ('clf',   MultinomialNB(alpha=1.0))  # alpha = Laplace smoothing
])

X_tr, X_te, y_tr, y_te = train_test_split(X_text, y_text, test_size=0.2, 
                                             random_state=42, stratify=y_text)
pipeline_mnb.fit(X_tr, y_tr)
y_pred = pipeline_mnb.predict(X_te)

print("=== Multinomial Naive Bayes: Spam Detection ===\n")
print(classification_report(y_te, y_pred, target_names=['Ham','Spam']))

fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_te, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Ham','Spam']).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Spam Detection')
plt.tight_layout()
plt.show()

In [ ]:
# Most discriminative words for spam vs ham
import re
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_vec = vectorizer.fit_transform(X_text)
mnb = MultinomialNB().fit(X_vec, y_text)

feature_names = vectorizer.get_feature_names_out()
# Log-ratio: how much more likely in spam vs ham
log_ratios = mnb.feature_log_prob_[1] - mnb.feature_log_prob_[0]

top_spam = pd.Series(log_ratios, index=feature_names).nlargest(15)
top_ham  = pd.Series(log_ratios, index=feature_names).nsmallest(15)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].barh(top_spam.index[::-1], top_spam.values[::-1], color='#e85d20', edgecolor='white')
axes[0].set_title('Top 15 Spam Words')
axes[0].set_xlabel('Log-ratio (spam/ham)')

axes[1].barh(top_ham.index, top_ham.abs().values, color='#1e5fa8', edgecolor='white')
axes[1].set_title('Top 15 Ham Words')
axes[1].set_xlabel('Log-ratio (ham/spam)')

plt.suptitle('Most Discriminative Words — Naive Bayes Spam Classifier', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print("\n📌 The model learned these patterns purely from labeled examples — no rules were written by hand")

In [ ]:
answers = {
    "q1": "",  # What is the conditional independence assumption in Naive Bayes?
    "q2": "",  # What type of Naive Bayes would you use for word counts in documents?
    "q3": "",  # What is Laplace smoothing and why is it needed?
    "q4": "",  # Why does Naive Bayes sometimes outperform logistic regression on small datasets?
    "q5": "",  # What is one real-world scenario where the independence assumption is clearly violated but NB still works?
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print("❌ Still empty: "+str(missing) if missing else "✅ Done! File → Save a copy in GitHub")

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "Naive Bayes"
# @title 🤖 AI-Graded Submission — fill in the box and click ▶ Run
# @markdown ---
# @markdown **Step 1:** Enter your GitHub username below
# @markdown
# @markdown **Step 2 (one-time):** Get a free AI grading key
# @markdown - Go to [aistudio.google.com](https://aistudio.google.com) — use your **personal Gmail** (not university email — many universities block AI Studio)
# @markdown - Click **Get API key → Create API key** → copy it
# @markdown - In Colab: click the **🔑 key icon** in the left sidebar → Add secret → Name: `GEMINI_API_KEY` → paste key → toggle ON
# @markdown - Done — this persists across all 30 notebooks automatically
# @markdown
# @markdown **Step 3:** Click ▶ Run on this cell for instant AI feedback

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

# ── DO NOT EDIT BELOW THIS LINE ──────────────────────────────────────────────
import os, json, textwrap, urllib.request, re as _re

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_gemini_key():
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            return key
    except Exception:
        pass
    return None

def _call_gemini(prompt, api_key):
    url = (f"https://generativelanguage.googleapis.com/v1beta/"
           f"models/gemini-2.0-flash:generateContent?key={api_key}")
    payload = json.dumps({
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"maxOutputTokens": 1024, "temperature": 0.3}
    }).encode()
    req = urllib.request.Request(
        url, data=payload,
        headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read())
            return data["candidates"][0]["content"]["parts"][0]["text"]
    except urllib.error.HTTPError as e:
        return f"API_ERROR:{e.code}:{e.read().decode()[:200]}"
    except Exception as e:
        return f"ERROR:{e}"

def _build_prompt(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    n_done   = len(answered)
    qa_lines = "\n".join(
        f"Q{k.replace('q','')}: {v}" for k, v in answered.items()
    )
    return f"""You are a helpful TA grading quiz answers for the \"{nb_name}\" notebook
in a data science course called \"Data Pattern Recognition for the Rest of Us\",
based on ISLP (Introduction to Statistical Learning with Python).

## Student Answers ({n_done}/{n_total} questions answered)
{qa_lines if qa_lines else "(none provided)"}

## Instructions
- Grade conceptual understanding, not exact wording
- Accept any reasonable paraphrase of the correct answer
- Be encouraging, specific, and concise
- Respond ONLY with valid JSON, no markdown fences, no extra text:

{{
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent | Good | Needs Review | Incomplete>",
  "feedback": "<2-3 sentences: what they got right, any misconceptions to fix>",
  "tip": "<one specific thing to remember or explore from this technique>"
}}"""

def _rule_based_grade(answers_dict):
    n_total    = len(answers_dict)
    answered   = [v for v in answers_dict.values() if str(v).strip()]
    n_answered = len(answered)
    avg_len    = sum(len(str(v)) for v in answered) / max(n_answered, 1)
    score      = n_answered
    if n_answered == 0:
        return {"quiz_score": 0, "grade": "Incomplete",
                "feedback": "No answers provided. Fill in the quiz answers above and re-run.",
                "tip": "Each question only needs 1-2 sentences."}
    elif avg_len < 15:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"You answered {n_answered}/{n_total} questions but most "
                             "responses are very brief. Try to explain your reasoning in 1-2 sentences."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback on your answers."}
    elif n_answered == n_total:
        return {"quiz_score": score, "grade": "Good",
                "feedback": (f"All {n_total} questions answered with good detail. "
                             "Add a free Gemini key for AI analysis of your specific reasoning."),
                "tip": "Get a free key at aistudio.google.com using your personal Gmail."}
    else:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"{n_answered} of {n_total} questions answered. "
                             f"Complete the remaining {n_total - n_answered} questions for full credit."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}

def _print_result(result, username, nb_name):
    colors = {
        "Excellent":    "\033[92m",
        "Good":         "\033[94m",
        "Needs Review": "\033[93m",
        "Incomplete":   "\033[91m",
    }
    R     = "\033[0m"
    grade = result.get("grade", "See feedback")
    C     = colors.get(grade, "\033[0m")
    n     = len(answers)
    qs    = result.get("quiz_score", 0)
    bar   = "\u2588" * qs + "\u2591" * (n - qs)

    print("\n" + "\u2550"*58)
    print(f"  \U0001f916  AI Feedback \u2014 {nb_name}")
    print("\u2550"*58)
    if username:
        print(f"  Student : @{username}")
    else:
        print(f"  Student : \u26a0\ufe0f  Enter your GitHub username in the box above")
    print(f"  Grade   : {C}{grade}{R}")
    print(f"  Score   : {qs}/{n}  [{bar}]")
    print()
    fb = result.get("feedback", "")
    for line in textwrap.wrap(fb, width=56):
        print(f"  {line}")
    tip = result.get("tip", "")
    if tip:
        print()
        for line in textwrap.wrap(f"\U0001f4a1 {tip}", width=56):
            print(f"  {line}")
    print("\u2550"*58 + "\n")

# ── Main grading flow ─────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    n_total    = len(answers)
    n_answered = sum(1 for v in answers.values() if str(v).strip())
    username   = GITHUB_USERNAME.strip()

    print(f"\n  Notebook : {_NB_TITLE}")
    print(f"  Answers  : {n_answered}/{n_total} provided")
    print(f"  Student  : @{username}" if username else
          "  Student  : \u26a0\ufe0f  Enter your GitHub username in the box above")

    api_key = _get_gemini_key()

    if api_key:
        print(f"  AI model : Gemini 2.0 Flash \u2713")
        print(f"  Grading  : please wait 10-20 seconds...\n")
        raw = _call_gemini(_build_prompt(answers, _NB_TITLE), api_key)
        if raw.startswith("API_ERROR") or raw.startswith("ERROR"):
            print(f"  \u26a0  {raw[:120]}")
            result = _rule_based_grade(answers)
        else:
            try:
                clean  = _re.sub(r"```(?:json)?\s*|```", "", raw).strip()
                result = json.loads(clean)
            except Exception:
                result = {"quiz_score": n_answered,
                          "grade": "Good" if n_answered == n_total else "Needs Review",
                          "feedback": raw[:400], "tip": ""}
    else:
        print("  AI model : rule-based (no Gemini key found)\n")
        print("  To enable AI grading:")
        print("  1. aistudio.google.com \u2192 Get API key (free, personal Gmail)")
        print("  2. Colab left sidebar \u2192 \U0001f511 Secrets")
        print("     Name: GEMINI_API_KEY  |  Value: your key  |  Toggle: ON")
        print("  3. Re-run this cell\n")
        result = _rule_based_grade(answers)

    _print_result(result, username, _NB_TITLE)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 choose your fork\n")


---
*Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*